# Rust 109: Logging

Chapter 09 of the Rust track, and the last of the core track — the chapter where the program finally *narrates itself*. Everything so far printed with `println!`/`eprintln!`; this chapter replaces that with **`tracing`**: timestamped, levelled, structured events and spans, filtered by configuration, fan-out to multiple sinks — and, in the second half, the honest Rust answer to Python's `logger.exception()`: errors that carry context chains, `std::backtrace::Backtrace`, and one logging call that prints message, chain and backtrace together.

This is also the **first crate chapter of the series**. Everything runs on `tracing` and friends, verified against this toolchain before a word was written (versions below). Two toolchain facts shape the whole chapter, and §0 states them plainly.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `09_logging_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer, arriving from Python and C++. You have used Loguru (or `logging`) enough to miss it: colourful stdout, file sinks, `logger.exception()` printing the whole traceback. That taste is the chapter's spec — and the chapter's job is to show what Rust's ecosystem answers, what it answers *differently*, and the one thing it cannot answer at all (interpreter-frame tracebacks): the differences are the content.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cell 2.1 is the subscriber setup the whole chapter relies on — run it first, and after any kernel restart.
- Outputs shown under cells are **plain-text renderings** of ANSI-coloured tracing output (the verifier strips ANSI); in a real terminal the levels are coloured.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic you would get. §7 is the catalogue.
- The prose is terse and the facts are dense. This is a reference you will come back to, not a novel you read once.

## The one idea to internalise

**Logging is not printing — it is structured, filterable telemetry fan-out.** An `info!(symbol = "AAPL", qty = 300, "order sent")` is not a line of text; it is *an event with typed fields*, which the active subscriber may render, filter, count, or ship to Loki. That decoupling — *emit everywhere, decide visibility elsewhere* — is what the `RUST_LOG` filter and the layer stack buy you. And the error path follows the same shape: capture the failure **with context** (`eyre`), capture *where* it happened (`Backtrace`), and hand the whole object to one `error!` event.


### Contents

- [0. Notebook Conventions: Crates and Kernel](#0-notebook-conventions-crates-and-kernel)
- [1. The Loguru to Rust Mapping](#1-the-loguru-to-rust-mapping)
- [2. Events: Levels and Fields](#2-events-levels-and-fields)
- [3. Spans: Context That Sticks](#3-spans-context-that-sticks)
- [4. The Subscriber: Layers, Filters, Sinks](#4-the-subscriber-layers-filters-sinks)
- [5. File Sinks: `tracing-appender` and the Worker Guard](#5-file-sinks-tracing-appender-and-the-worker-guard)
- [6. Errors That Carry Their Story with `eyre` and `Backtrace`](#6-errors-that-carry-their-story-with-eyre-and-backtrace)
- [7. Reading the Compiler: Panics and E-Codes](#7-reading-the-compiler-panics-and-e-codes)
- [8. Performance Notes for Low-Latency Code](#8-performance-notes-for-low-latency-code)
- [9. Mental Model: Emit vs Decide](#9-mental-model-emit-vs-decide)
- [10. Habits and Review](#10-habits-and-review)
- [11. Cheat Sheet](#11-cheat-sheet)
- [12. Review Questions](#12-review-questions)

If you are returning to this chapter later, §11 is the one-page summary, §7 is the diagnostics catalogue, and §6 is the `logger.exception()` answer.


## 0. Notebook Conventions: Crates and Kernel

This notebook runs on the **evcxr** kernel and is the first to load **crates** — the two facts below change how the cells behave, and both were verified against this exact toolchain (rustc 1.98.1, edition 2024) before the chapter was written.

### The crate list (verified versions)

| Crate | Version | Role |
|---|---|---|
| `tracing` | 0.1.44 | the macros: `info!`, `warn!`, spans, fields |
| `tracing-subscriber` | 0.3.23 | the subscriber side: fmt layer, `EnvFilter`, registry |
| `tracing-appender` | 0.2.5 | file sinks, rolling files, non-blocking workers |
| `eyre` | 0.6.14 | error reports with context chains (`anyhow`'s API twin) |

Load them with evcxr's `:dep` magic, **feature flags included** (a re-`:dep` with different features is a rebuild; changing features means restart the kernel):

```text
:dep tracing = "0.1"
:dep tracing-subscriber = { version = "0.3", features = ["env-filter"] }
:dep tracing-appender = "0.2"
:dep eyre = "0.6"
```

### Kernel facts that shape every cell

- **The kernel is one long-running process.** A global subscriber, once set, stays set for the whole session — `try_init` on a second attempt returns `Err` ("a global default trace dispatcher has already been set") — and the events keep flowing to the *first* subscriber; there is no un-set. Cells that need a specific stack use a **scoped dispatch** instead (`tracing::dispatcher::with_default`, §4.2, §5) — which never touches the global slot — and cell 2.1's `try_init` guard is what makes re-runs safe.
- **`RUST_LOG` arrives from the Jupyter server's environment** — likely unset — but `Backtrace::capture()` reads it **lazily, at the first call**: setting it *in a cell* genuinely enables capture (§6.2). First capture wins: before that first call, a cell can still set the variable.
- **ANSI is on by default** and Jupyter renders the colours; `with_ansi(false)` gives plain text (used in file cells).
- **`std::env::set_var` is `unsafe` in edition 2024** (chapter 07, §9) — the env cells carry the `unsafe` blocks.

Cells are written to be re-runnable top-to-bottom after a kernel restart.


## 1. The Loguru to Rust Mapping

Chapter 09 exists because the author's logging taste was shaped by **Loguru**. The mapping below is the chapter's contract — every row is taught, and the one that cannot be reproduced is called out honestly:

| Loguru habit | Rust idiom | This chapter |
|---|---|---|
| `logger.info(...)` and friends | `tracing::info!` and sibling macros | §2 |
| colourful formatted stdout | `tracing_subscriber::fmt` layer (ANSI default) | §2, §4 |
| file sink, rotation | `tracing_appender::rolling` + `non_blocking` | §5 |
| extra sinks (Loki, network) | additional `Layer`s (e.g. `tracing-loki`) | mentioned, §4 |
| `logger.exception()` + full traceback | **no direct equivalent** — errors carry context + `Backtrace`, logged with one event | §6 |
| `@logger.catch` decorator | nothing; explicit `Result` plus an `error!` event | §6 |

The structural reason for the one gap: Python runs on an interpreter that can walk live frames when an exception propagates; Rust compiles to machine code, frames inline, and a backtrace is **best-effort** — §6 shows exactly what that means on this toolchain (`Backtrace::capture` vs `force_capture`). `rust-loguru` exists but is not the teaching target: it cannot reproduce interpreter-level capture either, and the industry standard is `tracing`.


## 2. Events: Levels and Fields

An **event** is a point in time: something happened, at a level, with fields. The level macros — `error!`, `warn!`, `info!`, `debug!`, `trace!` — carry both the message and any number of `field = value` pairs. Fields are the structure: the subscriber can filter on them, format them, or ship them; `println!` can do none of that.

The default `fmt` look (ANSI stripped, one line per event):

```text
2026-09-25T06:48:56.298430Z  WARN order{symbol="AAPL" qty=300}: rebalance needed
2026-09-25T06:48:56.298347Z ERROR error-level event: 42
```

Timestamp, level (colour-coded in a terminal), target (the module path), message, fields. That single line is already richer than any `println!`.

### First events


In [ ]:
// 2.1 The chapter's subscriber: one cell, run once per kernel session.
// ANSI on (Jupyter renders it); EnvFilter reads RUST_LOG, defaulting to info.
:dep tracing = "0.1"
:dep tracing-subscriber = { version = "0.3", features = ["env-filter"] }
:dep tracing-appender = "0.2"
:dep eyre = "0.6"

use tracing_subscriber::{EnvFilter, fmt, prelude::*};
let filter = EnvFilter::try_from_default_env()
    .unwrap_or_else(|_| EnvFilter::new("info"));
let _ = tracing_subscriber::registry()
    .with(fmt::layer().with_target(false).with_filter(filter))
    .try_init(); // Err on a re-run — the guard that makes re-runs safe (no takeover, ever)
println!("subscriber ready (try_init Err on re-run is expected)");


In [ ]:
// 2.2 Events at every level; fields make them structured, not just text.
tracing::error!("book crossed: bid {} >= ask {}", 4126, 4125);
tracing::warn!("stale quote: {} ms old", 850);
tracing::info!(symbol = "AAPL", qty = 300, "order sent");
tracing::debug!("this is filtered out at the default info level");


### Field syntax: plain, `%` and `?`

Event fields take three shapes: `field = value` records the value *structurally*; `field = %value` records its `Display` form; `field = ?value` records its `Debug` form. The percent sigil is how an error report — whose whole value *is* its rendered chain — enters a structured event.

### Recording values three ways


In [ ]:
// 2.3 Plain fields are structural; % renders Display; ? renders Debug.
let latency_ms = 42u64;
let symbol = "AAPL";
tracing::info!(latency_ms, symbol, "plain fields use the variable's name");
tracing::info!(price = 4125.5, qty = 300, "typed fields");
tracing::warn!(detail = %format!("stale by {}ms", 850), "display form");
tracing::error!(err = ?std::io::Error::new(std::io::ErrorKind::NotFound, "feed file"), "debug form");


Note the first line's shorthand: `latency_ms, symbol` with no `=` records each variable under its own name — the everyday spelling. The `?`-form of the `io::Error` renders its `Debug` (`Os { code: 2, ... }`), which chapter 07's panic catalogue already taught you to read.

## 3. Spans: Context That Sticks

An **event** is a point; a **span** is an interval — "this work happened between here and here". Spans carry fields too, and every event logged *inside* the span is stamped with it: the context stops being repeated per line and becomes part of the record.

```text
2026-09-25T06:48:56.298430Z  WARN order{symbol="AAPL" qty=300}: rebalance needed
```

That `order{symbol="AAPL" qty=300}:` prefix is span context — logged once on the span, attached to every event within.

### Entering a span


In [ ]:
// 3.1 A span with fields; events inside carry the context automatically.
{
    let _order = tracing::info_span!("order", symbol = "AAPL", qty = 300).entered();
    tracing::info!("submitting");
    tracing::warn!("partial fill: 120 of 300");
} // the guard dropped: span exited
tracing::info!("outside the span: no order context anymore");


The `.entered()` guard is the teaching form: the span is active exactly while the guard lives, and the `{ .. }` block makes that lifetime visible. (The macro-attribute form `#[instrument]` from `tracing-attributes` does the same for functions — it needs the `attributes` feature of `tracing`, which is default-on; this chapter stays with explicit guards because the kernel has no functions to decorate.)

### Spans nest

Spans form a tree: entering a span inside another span stacks the context, and the fmt layer prints the whole chain. A session span wrapping order spans is the running example:

### A session span wrapping two orders


In [ ]:
// 3.2 Nested spans: the fmt layer prints the full context chain.
{
    let _session = tracing::info_span!("session", id = 7).entered();
    tracing::info!("session started");
    {
        let _order = tracing::info_span!("order", symbol = "MSFT", qty = 120).entered();
        tracing::info!("submitting");
    }
    {
        let _order = tracing::info_span!("order", symbol = "AAPL", qty = 300).entered();
        tracing::warn!("rebalance needed");
    }
}


## 4. The Subscriber: Layers, Filters, Sinks

Nothing you have seen so far was printed by `tracing` itself — `info!` only *emits*. The **subscriber** decides what happens: the `Registry` collects **layers**, each layer sees every event and span and does its own thing (format to stdout, write to a file, ship to a network sink). Emit-vs-decide is the chapter's central split, and this is the "decide" half:

| Layer | Job | Chapter cell |
|---|---|---|
| `fmt::layer()` | render events (ANSI, timestamps) | 2.1 |
| `EnvFilter` | choose which events exist, from `RUST_LOG` | 4.1 |
| `tracing_appender` file writer | a second sink | §5 |
| `tracing-loki` (not loaded here) | ship to Grafana Loki | mentioned only |

### `EnvFilter`: the `RUST_LOG` grammar

`EnvFilter` parses the same directives Loguru users configure in code: `RUST_LOG=info` (everything at info and up), `RUST_LOG=my_app=debug,other=warn` (per-target levels), `RUST_LOG=warn` (the quiet default). The filter is consulted at *emit time*, so re-running a cell with a different filter changes what passes — as long as the subscriber is rebuilt.

### One filter, three configurations


In [ ]:
// 4.1 EnvFilter directives: the RUST_LOG grammar, exercised directly.
// (EnvFilter is already in scope from cell 2.1.)
/// The coarsest level an event may have and still pass the directive string.
fn passes(directives: &str) -> Option<tracing_core::LevelFilter> {
    EnvFilter::try_new(directives).unwrap().max_level_hint()
}
for (d, name) in [("info", "RUST_LOG=info"), ("warn", "RUST_LOG=warn"), ("feed=debug,app=warn", "per-target")] {
    println!("{name:<12} -> max level hint: {:?}", passes(d));
}
assert_eq!(passes("info"), Some(tracing_core::LevelFilter::INFO));
assert_eq!(passes("warn"), Some(tracing_core::LevelFilter::WARN));
assert_eq!(passes("feed=debug,app=warn"), Some(tracing_core::LevelFilter::DEBUG));


`would_enable` is the filter's own question-answering API — the same logic the subscriber runs per event, surfaced here so the grammar is checkable in a cell. The per-target form is the one that scales in real projects: noisy dependencies at `warn`, your own crate at `debug`, all without recompiling.

### A second layer joins: one event, two destinations

Layers compose on the registry. The next cell adds a *second* fmt layer writing to an in-memory buffer with a stricter filter — one `info!` call, and both the stdout layer and the buffer layer see it. That is the whole fan-out story in one cell:

### Fan-out: one event, two layers


In [ ]:
// 4.2 Fan-out without touching the global slot: a scoped dispatch.
// (tracing::dispatcher::with_default installs a subscriber for the closure
// only - the kernel-safe way to demonstrate any layer stack.)
use std::sync::{Arc, Mutex};
let buffer: Arc<Mutex<Vec<String>>> = Arc::new(Mutex::new(Vec::new()));
let sink = buffer.clone();
let scoped = tracing::Dispatch::new(
    tracing_subscriber::registry()
        .with(
            tracing_subscriber::fmt::layer()
                .with_ansi(false)
                .with_target(false)
                .with_filter(tracing_subscriber::EnvFilter::new("info")),
        )
        .with(
            tracing_subscriber::fmt::layer()
                .with_ansi(false)
                .with_target(false)
                .with_writer(move || MemWriter(sink.clone()))
                .with_filter(tracing_subscriber::EnvFilter::new("info")),
        ),
);
tracing::dispatcher::with_default(&scoped, || {
    tracing::warn!("fan-out probe: every layer of the scoped stack sees this");
});
// back on the global subscriber here - nothing was replaced, nothing leaked
let captured = buffer.lock().unwrap().join("
");
println!("buffer captured {} line(s)", captured.lines().count());
assert!(captured.contains("fan-out probe"), "buffer holds: {captured}");

/// A writer that appends every formatted line into a shared Vec - the
/// whole "custom sink" story (Loki is this shape pointed at a network).
struct MemWriter(Arc<Mutex<Vec<String>>>);
impl std::io::Write for MemWriter {
    fn write(&mut self, buf: &[u8]) -> std::io::Result<usize> {
        let text = String::from_utf8_lossy(buf).trim().to_string();
        if !text.is_empty() {
            self.0.lock().unwrap().push(text);
        }
        Ok(buf.len())
    }
    fn flush(&mut self) -> std::io::Result<()> { Ok(()) }
}


Cell 4.2 is the Loki story in miniature: a sink is just "something that implements `Write` and wants events" — `tracing-loki` is that same shape pointed at a network. Two setup tools, two jobs. The **global** subscriber (`try_init`/`init`) is set once per process and never replaced — a refused second `try_init` keeps the old stack receiving events, and `init` on an occupied slot panics (§7). That is why this cell uses a **scoped dispatch**: `tracing::dispatcher::with_default` installs the stack for the closure only, then hands control back to the global — the kernel-safe way to demonstrate any layer combination. Note what the buffer proves: the event inside the scope hit *both* layers, and the same event after the scope would hit neither of them.


## 5. File Sinks: `tracing-appender` and the Worker Guard

`tracing_appender` is two features in one crate: **rolling files** (hourly/daily/never rotation, `logs/` prefix naming) and **non-blocking writers** — the file I/O happens on a background worker thread, and the hot path only enqueues. Two API rules make the segment:

1. `non_blocking(writer)` returns a **pair**: the writer to hand the layer, and a `WorkerGuard` that **must be kept alive** — dropping it shuts the worker down *and flushes*, so its lifetime defines what the file actually receives.
2. In `main`, the file stack is installed with plain `init()` and the guard bound in `main` itself. In this notebook the global slot is already taken (§2.1) and cannot be replaced — so the file cells use a **scoped dispatch** (§4.2's `with_default`) and keep the guard inside the scope. Same APIs, kernel-safe lifetime.

### The daily-rotation table


In [ ]:
// 5.1 A rolling file sink behind a scoped dispatch; the WorkerGuard stays alive.
let dir = "scratch09";
std::fs::create_dir_all(dir).unwrap();
let file = tracing_appender::rolling::never(dir, "session.log");
let (writer, guard) = tracing_appender::non_blocking(file);
let file_scoped = tracing::Dispatch::new(
    tracing_subscriber::registry()
        .with(
            tracing_subscriber::fmt::layer()
                .with_ansi(false)
                .with_target(false)
                .with_writer(writer)
                .with_filter(tracing_subscriber::EnvFilter::new("info")),
        ),
);
// The guard must outlive every event the sink should receive - here it is
// bound INSIDE the scope, so the dispatch cannot outlive its worker.
tracing::dispatcher::with_default(&file_scoped, || {
    let _guard = guard; // keep the worker alive for the scope's events
    tracing::info!("file sink event: enqueued through the non-blocking worker");
});
tracing::info!("(this event goes to the global subscriber, not the file)");
println!("event enqueued; flush happens at the guard drop (next cell)");


In [ ]:
// 5.2 The flush boundary: drop the guard, then read the file - the event is there.
std::fs::create_dir_all("scratch09").unwrap();
let file = tracing_appender::rolling::never("scratch09", "flush.log");
let (writer, guard) = tracing_appender::non_blocking(file);
let file_scoped = tracing::Dispatch::new(
    tracing_subscriber::registry()
        .with(
            tracing_subscriber::fmt::layer()
                .with_ansi(false)
                .with_target(false)
                .with_writer(writer)
                .with_filter(tracing_subscriber::EnvFilter::new("info")),
        ),
);
tracing::dispatcher::with_default(&file_scoped, || {
    let guard = guard; // shadowed so THIS cell owns the drop
    tracing::warn!("flush probe: written, then the guard drops");
    drop(guard); // worker shuts down AND flushes - while still inside the scope
});
let content = std::fs::read_to_string("scratch09/flush.log").unwrap_or_default();
println!("file received the event: {}", content.contains("flush probe"));
assert!(content.contains("flush probe"));
let _ = std::fs::remove_dir_all("scratch09");


Cells 5.1–5.2 are one lesson split in two: **the guard's lifetime is the flush boundary**. 5.1 enqueues and keeps its guard alive for the scope; 5.2 drops *its* guard explicitly — and only then is the file readable with the event in it. In a real binary the guard lives in `main`; in the kernel, the scope plays main's role. The rolling constructors (`rolling::daily`, `rolling::hourly`) produce the same writer type with a dated filename — `never` keeps the demo deterministic.

## 6. Errors That Carry Their Story with `eyre` and `Backtrace`

This is the `logger.exception()` answer, and it has three parts: the **context chain** (`eyre::Report`), the **backtrace** (`std::backtrace::Backtrace`), and the **one event** that prints both. None of the three requires a crate beyond `eyre` — `Backtrace` is `std`.

### The context chain

`eyre::eyre!("...")` creates a report from a message; `.wrap_err("...")` wraps it with higher-level context. `Display` prints only the outermost message; **`{:?}` prints the chain** — outermost first, causes numbered:

### Building and rendering a chain


In [ ]:
// 6.1 The context chain: Display is the headline; Debug is the whole story.
use std::backtrace::Backtrace;
// wrap_err is an inherent method on eyre::Report - no trait import needed
// (the eyre::WrapErr trait is for wrapping generic Results)
let report: eyre::Report = eyre::eyre!("root cause: connection reset")
    .wrap_err("feed reconnect")
    .wrap_err("market data session");
println!("--- Display (what a user sees) ---");
println!("{report}");
println!("--- Debug (what a log file wants) ---");
println!("{report:?}");
assert!(format!("{report:?}").contains("Caused by:"));
assert!(format!("{report:?}").contains("0: feed reconnect"));


The `Caused by:` block numbered `0, 1, ...` is the `anyhow`/`eyre` house style — read it bottom-up (root cause last) or top-down (operator context first). The `Location:` line is free of charge: `eyre` stamps where the report was created.

### The backtrace, honestly

`std::backtrace::Backtrace::capture()` is **gated**: it captures only when `RUST_BACKTRACE` is set, and reports its own decision via `.status()`. Probed on this toolchain, two facts matter for the kernel:

- `capture()` consults the gate **once per process, at the first call** — set `RUST_BACKTRACE` before that first capture to enable it for the session; nothing later can change the cached answer;
- `Backtrace::force_capture()` ignores the gate entirely — status `Captured` even with nothing set — and on this machine yields ~40 frames with symbol names, including `main`.

The gap versus Loguru remains real: frames show `spike09::main`-style symbols, not source-level line-at-a-time interpreter frames. Best-effort is the honest phrase, and ~40 usable frames is what best-effort means here.

### The gate, exercised


In [ ]:
// 6.2 capture() vs force_capture(): the RUST_BACKTRACE gate.
// capture()'s answer depends on the session's history (the gate is cached at
// the process's FIRST capture call), so this cell asserts only the half that
// is deterministic: force_capture() ignores the gate entirely.
// (Backtrace is already in scope from 6.1.)
let gated = Backtrace::capture();
println!("capture() says: {:?}", gated.status()); // Disabled or Captured - session history decides
let forced = Backtrace::force_capture();
println!("force_capture() says: {:?}", forced.status());
assert!(matches!(forced.status(), std::backtrace::BacktraceStatus::Captured));
let forced_text = format!("{forced}");
println!("forced frames: {}", forced_text.lines().count());
assert!(forced_text.contains("main"));
println!("(kernel rule: set RUST_BACKTRACE BEFORE the session's first capture)");


The gate is not read per call — it is **cached from the process's very first `Backtrace::capture()`**, and the cached value wins for the whole session (probed four ways while writing this chapter: unset, set, removed, and `=0`, in both orders). Consequences for the kernel: set `RUST_BACKTRACE` *before* the session's first capture and every later `capture()` is enabled; if the first capture already happened disabled, no in-session `set_var` can turn it on — restart the kernel. That is why cell 6.2 asserts only the `force_capture()` half: the `capture()` half depends on the session's history, and asserting it would make the cell's result depend on which cells ran before it.

### The Loguru `exception()` pattern, assembled

One error event, three payloads: the message, the chain, the backtrace. This is the whole answer:

### The assembled pattern


In [ ]:
// 6.3 The logger.exception() equivalent: message + chain + backtrace in one event.
// (Backtrace and WrapErr already in scope from 6.1)
/// Simulate a feed failure and return the report plus where it happened.
fn connect_to_feed() -> eyre::Report {
    eyre::eyre!("root cause: connection reset")
        .wrap_err("feed reconnect")
        .wrap_err("market data session")
}
let report = connect_to_feed();
let bt = Backtrace::force_capture();
tracing::error!(
    error = %report,
    backtrace = %bt,
    "pipeline failed" // the message, Loguru-style
);
println!("(the event above shows: message, then error chain, then backtrace)");


In a terminal the `backtrace = %bt` field is a multi-line block under the event — the visual analogue of `logger.exception()`'s traceback, produced by an explicit field rather than magic. **Panic path:** the same two tools cover it — chapter 07's `catch_unwind` payload plus `Backtrace::force_capture()` in the hook is the panic analogue, and chapter 11 will formalise the error types behind these reports.

For release builds: debug binaries symbolise as shown; stripped or optimised builds may inline frames away — the honest caveat from the spec, stated once.


## 7. Reading the Compiler: Panics and E-Codes

The crate world adds one runtime behaviour and a handful of compile errors that all say the same thing from different angles: *the global subscriber is a process-wide singleton*.

| Diagnostic | Trigger | Message (abridged) | Fix |
|---|---|---|---|
| **panicked: `SetGlobalDefaultError`** | `init()` when a subscriber exists | ``failed to set global default subscriber: a global default trace dispatcher has already been set`` | restart the kernel; there is no un-set — design cells so the *first* setup uses `try_init`, takeovers run once per session |
| **E0433** | `use tracing_subscriber;` without the crate loaded | ``cannot find crate`` (the `:dep` cell was not run) | run the `:dep` cell first; after a kernel restart, from the top |
| **E0432** | `use tracing_subscriber::EnvFilter;` without the `env-filter` feature | ``unresolved import`` — the module does not exist feature-less | `:dep tracing-subscriber = { version = "0.3", features = ["env-filter"] }` |
| **E0599** | `.with_filter(..)` on a bare layer | ``no method named `with_filter` found`` | the filtered-layer combinators live behind `tracing_subscriber::prelude::*` |
| **E0599** | `Backtrace::of_value`-style guessing | ``no associated function ... found`` | the constructors are `capture()` and `force_capture()` |

The `SetGlobalDefaultError` *panic* is the runtime twin of the redefinition error from chapter 01's §0: the kernel session is one process, and process-global state does not reset between cells. Every setup cell in this chapter uses `try_init` + `let _ =` for exactly this reason.

### Seeing the singleton error, safely


In [ ]:
// 7.1 The double-init behaviour, caught: try_init errs, init panics.
let second = tracing_subscriber::registry()
    .with(fmt::layer().with_ansi(false).with_filter(EnvFilter::new("info")));
match second.try_init() {
    Ok(_) => println!("(this session had no subscriber yet — cell 2.1 was not run)"),
    Err(e) => println!("try_init refused a second subscriber: {e}"),
    // and the refusal is final: events still flow to the FIRST subscriber.
}
// fmt().init() would have PANICKED here with SetGlobalDefaultError.


## 8. Performance Notes for Low-Latency Code

Logging in a hot loop is the classic latency tax. The crate family has the same answer at every layer:

| Habit | Cost shape | Fix |
|---|---|---|
| `info!` per tick, filtered at emit time | the event is *built* (fields, metadata) before the filter can reject it cheaply — still a call, an atomic level check, and a no-op dispatch | gate by level *and* keep hot events at `debug!`/`trace!` so release `RUST_LOG` runs them never |
| blocking file writer in the hot path | every event pays disk latency | `tracing_appender::non_blocking` — the hot path enqueues, a worker writes |
| forgetting the `WorkerGuard` | events silently lost at shutdown | the guard lives as long as the sink matters (§5) |
| `force_capture()` in a hot path | ~40 frames of stack walk per call | capture on the **error** path only, never per tick |
| ANSI formatting on a file sink | colour codes in log files | `with_ansi(false)` for files (§5 did), keep ANSI for terminals |

One design rule from the HFT seat: **log at boundaries, not inside loops.** The per-order span (§3) with one entry event and one outcome event tells the whole story; ten events per iteration tell it ten times slower.

## 9. Mental Model: Emit vs Decide

| Half | Who | What it owns |
|---|---|---|
| **Emit** | library and application code | `info!`/`warn!`/`error!` calls, spans, fields — *structured facts* |
| **Decide** | the installed subscriber | visibility (`EnvFilter`), rendering (`fmt`), destinations (stdout/file/Loki), sampling |

The split is why the same binary is chatty in development (`RUST_LOG=debug`) and quiet in production (`RUST_LOG=warn`) without a rebuild — the emit half never changed. It is also why `logger.exception()` has no *macro* equivalent: the exception handler in Python is both emitter and formatter, while Rust's emit half hands a **structured error object** (chain + backtrace) to the decide half, which formats it. More moving parts, and *more* power: the same `error!` event feeds stdout, a file, and Loki.

## 10. Habits and Review

- **`try_init` for the first setup cell; `init` only for a deliberate, once-per-session takeover** — a refused second subscriber keeps receiving events, and `init` on an occupied slot panics (§0, §4.2, §7).
- **Fields over strings:** `info!(symbol, qty, "sent")` beats `info!("sent AAPL x300")` — the structure survives filtering (§2).
- **Spans for context, events for moments** — the context is logged once on the span, not repeated per event (§3).
- **`RUST_LOG` per target in real projects:** `feed=debug,app=warn` (§4).
- **The `WorkerGuard` outlives the last event** — its drop is the flush point (§5).
- **`%report` for errors, `{:?}` to read the chain by hand** (§6).
- **`capture()` when the gate may be on, `force_capture()` when the story matters** — and never in a hot path (§6, §8).
- **Log at boundaries** — span per unit of work, one event per outcome (§8).

## 11. Cheat Sheet

| Task | Code |
|---|---|
| load crates (once per kernel) | `:dep tracing = "0.1"` (+ subscriber with `env-filter`, appender, eyre) |
| subscriber (re-run safe) | `fmt::layer().with_target(false).try_init()` behind `let _ =` |
| plain subscriber | `tracing_subscriber::fmt().init()` |
| event with fields | `info!(symbol, qty, "sent")` / `info!(price = 4125.5, "tick")` |
| Display / Debug field | `error!(err = %report, "..")` / `error!(err = ?value, "..")` |
| span with context | `let _g = info_span!("order", symbol = "AAPL").entered();` |
| filter grammar | `RUST_LOG=info` · `feed=debug,app=warn` |
| file sink | `rolling::never(dir, "app.log")` → `non_blocking` → layer writer |
| flush point | dropping the `WorkerGuard` |
| context chain | `eyre!("root").wrap_err("context")`; read with `{e:?}` |
| gated backtrace | `Backtrace::capture()` — needs `RUST_BACKTRACE` set before the first call |
| ungated backtrace | `Backtrace::force_capture()` |
| the Loguru pattern | `error!(error = %report, backtrace = %bt, "pipeline failed")` |
| double init | `try_init()` → `Err` (safe); `init()` → panic |

## 12. Review Questions

Answer from memory; each one is a §-numbered cell away.

1. Why does `info!(symbol = "AAPL", qty = 300, "sent")` beat `info!("sent AAPL x300")` when the subscriber is the one deciding visibility? (§2, §9)
2. Cell 3.2 logged `order{...}` context on events from two different orders. What would the flat-`println!` version of that log look like, and what would you have to repeat? (§3)
3. `RUST_LOG=feed=debug,app=warn` — which events from your own code survive, and what does this buy over a single global level? (§4)
4. Why must the `WorkerGuard` binding outlive the last event you care about, and what exactly happens at its drop? (§5)
5. `Display` vs `{:?}` on an `eyre::Report` — which one does an operator see, which one belongs in the log file, and why does the chapter log with `%report`? (§6)
6. You removed `RUST_BACKTRACE` after one enabled `capture()`. What does the next `capture().status()` read, and what does that tell you about when the gate is consulted? (§6)
7. `try_init()` returned `Err` in cell 4.2 and the notebook carried on. What would `init()` have done, and what does that make the global subscriber? (§7)
8. Where would a `force_capture()` call be defensible in a trading system, and where is it a latency bug? (§6, §8)

### What Comes Next

Chapter 09 closes the **core track**: the language, its ownership model, the standard library's surfaces, and now the program's ability to narrate itself. What follows is the applied track — and the first thing it does is stop pretending programs have no dependencies.

| Chapter | Title | What it settles |
|---|---|---|
| **10** | **Structs, OOP and Traits** | your own types — including the `Debug`/`Display` impls this chapter's fields kept rendering |
| 11 | Errors in Depth | real error types behind the `eyre::Report`s this chapter logged |
| 15 | Data Types and Their Libraries | the next crate chapter — the pattern `:dep` established here repeats there |

**Carry forward.** You should now be able to do these without looking anything up:

- set up a filtered, non-panicking subscriber in one cell and know why the guard exists
- emit structured events with plain/`%`/`?` fields and wrap work in spans
- read and write `RUST_LOG` directives, including per-target levels
- attach a rolling file sink through `non_blocking` and say where the flush happens
- build an `eyre` chain, capture a backtrace (gated and forced), and log both in one event
- name the one thing Rust logging cannot reproduce, and what it does instead
